# CodeBERT SmartBugs classifier: re-run protocol (v3)

This notebook re-runs the neural evaluation with every correction requested by Dr Siraj and produces
the CodeBERT findings needed for the full three-engine run. It is a re-run kit: the numbers in the
current report come from the previous notebook and must not be replaced until this run has finished
and its outputs have been checked.

| Dr Siraj point | What this notebook does |
|---|---|
| 1 | Trains a CodeBERT model with the three proof-of-concept (PoC) contracts **excluded** and writes its findings on those contracts in the common schema (`codebert_findings_poc.json`), so Slither + SmartCheck + CodeBERT can be aggregated and passed to the testnet validator. |
| 3 | Majority-class baseline; MCC and macro-F1 first; macro-F1 over all classes and over classes with at least `MIN_SUPPORT` examples; macro precision/recall and Cohen's kappa per fold and pooled. |
| 3 | Checkpoint selection on an **inner, contract-grouped validation split** of the training folds, never on the outer test fold. |
| 3 | **Truly nested** per-class thresholds (chosen on the inner validation predictions of each outer fold's model) plus the earlier cross-fitted version for comparison; the multi-class decision rule (Eq. 3.24) and overall macro-F1 / MCC / kappa after thresholding. |
| 3 | Optional plain cross-entropy ablation (`LOSS="ce"`, `CLASS_WEIGHTS=False`) so the effect of focal loss and class weighting can be measured. |
| 3 | Rare-class policy (`keep`, `merge` into *other*, or `exclude`). |
| 3 | Duplicate screening: exact duplicates of normalised function source across contracts are reported, and their contracts are merged into one group so copies cannot cross a split. |
| 3 | Several seeds, and **all out-of-fold probabilities saved** to disk. |
| 4 | Exact versions written to `environment.json`; fixes the metadata bug in which `thresholds_nested` stored F1 values instead of thresholds. |

Run on Kaggle with a GPU: attach the dataset that contains `functions.csv` (and, for Step 9, a folder
with the three PoC `.sol` files), edit the CONFIG cell, then Run All. One seed with 5 folds takes
roughly as long as the previous run; three seeds take about three times as long.

In [ ]:
# =============================== CONFIG ===============================
from pathlib import Path
CONFIG = dict(
    MODEL_NAME      = "microsoft/codebert-base",
    MODEL_REVISION  = None,          # optionally pin a Hugging Face commit hash for Appendix E
    DATA_CSV        = "/kaggle/input/smartbugs-functions/functions.csv",
    POC_DIR         = "/kaggle/input/poc-contracts",   # folder with the 3 PoC .sol files (Step 9)
    OUTPUT_DIR      = "/kaggle/working",
    EXPORT_DIR      = "/kaggle/working/codebert-smartbugs-v3",

    # contract_id values (e.g. "reentrancy/simple_dao.sol") of any PoC contract that is ALSO in SmartBugs
    # Curated; those are excluded from training. Leave empty if the PoC contracts are not in the corpus.
    POC_CONTRACT_IDS = [],

    # If functions.csv is not attached, the notebook rebuilds it from SmartBugs Curated at this commit
    # (needs Settings > Internet ON, or attach the smartbugs-curated repository as a Kaggle dataset).
    SMARTBUGS_REPO   = "https://github.com/smartbugs/smartbugs-curated.git",
    SMARTBUGS_COMMIT = "230e649123477eff332742a59a1c7cc6dc286cab",

    RUN_TAG         = "focal_weighted",   # change to "ce_unweighted" for the ablation run
    LOSS            = "focal",            # "focal" or "ce"
    CLASS_WEIGHTS   = True,               # inverse-frequency alpha (benign = 1)
    SEEDS           = [42, 43, 44],
    N_SPLITS        = 5,
    INNER_VAL_FRAC  = 0.2,                # share of training contracts held out for early stopping + thresholds
    MAX_LEN         = 256,
    EPOCHS          = 15,
    PATIENCE        = 3,
    BATCH_SIZE      = 8,
    GRAD_ACCUM      = 4,
    LR              = 2e-5,
    WEIGHT_DECAY    = 0.01,
    WARMUP_RATIO    = 0.10,
    FOCAL_GAMMA     = 2.0,
    MSD_K           = 5,
    DROPOUT         = 0.3,
    FP16            = True,

    MIN_SUPPORT     = 5,                  # classes below this are 'rare'
    RARE_POLICY     = "keep",             # "keep" | "merge" (into 'other') | "exclude"
    DEDUP_GROUPS    = True,               # merge contracts sharing an identical normalised function
    DEDUP_MIN_CHARS = 60,                 # ignore trivial functions (getters etc.) when screening

    COL_SOURCE="source", COL_LABEL="label", COL_FUNC_LABEL="func_label", COL_GROUP="contract_id",
    BENIGN="benign",
)
Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["EXPORT_DIR"]).mkdir(parents=True, exist_ok=True)
OUT = CONFIG["OUTPUT_DIR"]; TAG = CONFIG["RUN_TAG"]
CONFIG

In [ ]:
# ===================== environment + reproducibility =====================
import os, random, json, platform, hashlib, re, glob, copy
import numpy as np, pandas as pd, torch
import transformers, sklearn

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VERSIONS = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__,
                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                model=CONFIG["MODEL_NAME"], model_revision=CONFIG["MODEL_REVISION"])
json.dump(VERSIONS, open(f"{OUT}/environment.json", "w"), indent=2)
print(json.dumps(VERSIONS, indent=2))

## 1. Data, labels and duplicate screening

Functions with no annotated vulnerable line are labelled benign (`func_label` empty or `benign`).
That convention is kept, and stated in the report as such.

If `functions.csv` is not attached, the next cell rebuilds it from SmartBugs Curated: every function,
constructor, fallback and modifier body is extracted with its line range, each annotated line in
`vulnerabilities.json` is mapped to the enclosing body, and every other body is labelled benign. At the
pinned commit this gives exactly 993 bodies from 143 contracts with the class counts of Table 3.1.

In [ ]:
import subprocess
def _strip_comments(s):
    out = []; i = 0; n = len(s)
    while i < n:
        if s.startswith("//", i):
            j = s.find("\n", i); j = n if j == -1 else j; out.append(" " * (j - i)); i = j
        elif s.startswith("/*", i):
            j = s.find("*/", i + 2); j = n if j == -1 else j + 2
            out.append("".join(c if c == "\n" else " " for c in s[i:j])); i = j
        elif s[i] in "\"'":
            q = s[i]; j = i + 1
            while j < n and s[j] != q:
                if s[j] == "\\": j += 1
                j += 1
            j = min(j + 1, n)
            out.append(q + "".join(c if c == "\n" else " " for c in s[i + 1:j - 1]) + q if j - i >= 2 else s[i:j]); i = j
        else:
            out.append(s[i]); i += 1
    return "".join(out)

_HDR = re.compile(r"\b(function\b\s*(\w*)|constructor\b|modifier\s+(\w+)|fallback\b|receive\b)")
def sol_bodies(src):
    clean = _strip_comments(src); lines = src.split("\n"); res = []
    for m in _HDR.finditer(clean):
        kind = m.group(1)
        if kind in ("fallback", "receive", "constructor") and not re.match(r"\s*\(", clean[m.end():]): continue
        b = clean.find("{", m.end()); sc = clean.find(";", m.end())
        if b == -1 or (sc != -1 and sc < b): continue
        d = 0; j = b
        while j < len(clean):
            if clean[j] == "{": d += 1
            elif clean[j] == "}":
                d -= 1
                if d == 0: break
            j += 1
        name = m.group(2) or m.group(3) or kind.split()[0]
        if kind.startswith("function") and not m.group(2): name = "fallback"
        sl = clean.count("\n", 0, m.start()) + 1; el = clean.count("\n", 0, j) + 1
        res.append(dict(func_name=name, start_line=sl, end_line=el, source="\n".join(lines[sl - 1:el])))
    return res

def build_functions_csv(repo_dir, out_csv):
    V = json.load(open(os.path.join(repo_dir, "vulnerabilities.json")))
    rows, outside, conflicts = [], 0, 0
    for c in V:
        src = open(os.path.join(repo_dir, c["path"]), encoding="utf-8", errors="ignore").read()
        fs = sol_bodies(src); lab = [set() for _ in fs]
        for v in c["vulnerabilities"]:
            for ln in v["lines"]:
                hit = [k for k, f in enumerate(fs) if f["start_line"] <= ln <= f["end_line"]]
                if not hit: outside += 1; continue
                lab[min(hit, key=lambda k: fs[k]["end_line"] - fs[k]["start_line"])].add(v["category"])
        for f, l in zip(fs, lab):
            conflicts += len(l) > 1
            y = sorted(l)[0] if l else "benign"
            rows.append(dict(contract_id=c["path"].replace("dataset/", ""), func_name=f["func_name"],
                             start_line=f["start_line"], end_line=f["end_line"], func_label=y, label=y, source=f["source"]))
    out = pd.DataFrame(rows); out.to_csv(out_csv, index=False)
    print(f"built {out_csv}: {len(out)} functions, {out.contract_id.nunique()} contracts, "
          f"{outside} annotated lines outside any body, {conflicts} conflicting labels")
    print(out.func_label.value_counts().to_dict())
    return out_csv

def smartbugs_repo():
    hits = glob.glob("/kaggle/input/**/vulnerabilities.json", recursive=True)
    if hits: return os.path.dirname(hits[0])
    dest = os.path.join(CONFIG["OUTPUT_DIR"], "smartbugs-curated")
    if not os.path.exists(dest):
        r = subprocess.run(["git", "clone", CONFIG["SMARTBUGS_REPO"], dest], capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-500:]); return None
    subprocess.run(["git", "-C", dest, "checkout", "-q", CONFIG["SMARTBUGS_COMMIT"]])
    return dest

In [ ]:
C = CONFIG
NEEDED = {C["COL_SOURCE"], C["COL_GROUP"]}

def _has_cols(path):
    try:
        return NEEDED.issubset(pd.read_csv(path, nrows=2).columns)
    except Exception:
        return False

def find_data_csv(configured):
    # 1) the configured path; 2) any functions.csv under /kaggle/input; 3) any CSV with the needed columns
    if os.path.exists(configured):
        return configured
    roots = ["/kaggle/input", os.getcwd()]
    hits = [p for r in roots for p in glob.glob(os.path.join(r, "**", "functions.csv"), recursive=True)]
    if hits:
        return sorted(hits)[0]
    for r in roots:
        for p in sorted(glob.glob(os.path.join(r, "**", "*.csv"), recursive=True)):
            if _has_cols(p):
                return p
    return None

def show_inputs(root="/kaggle/input", depth=4):
    for dirpath, dirnames, files in os.walk(root):
        level = dirpath[len(root):].count(os.sep)
        if level >= depth:
            dirnames[:] = []
            continue
        print("  " * level + os.path.basename(dirpath) + "/")
        for f in files[:15]:
            print("  " * (level + 1) + f)

resolved = find_data_csv(C["DATA_CSV"])
if resolved is None:
    print("functions.csv not attached: rebuilding it from SmartBugs Curated ...")
    repo = smartbugs_repo()
    if repo is not None:
        resolved = build_functions_csv(repo, os.path.join(C["OUTPUT_DIR"], "functions.csv"))
if resolved is None:
    print("Attached inputs under /kaggle/input:")
    show_inputs()
    raise FileNotFoundError(
        "Could not find functions.csv (or any CSV with columns %s).\n"
        "Fix: either turn on Settings > Internet so the notebook can clone SmartBugs Curated, or attach "
        "functions.csv (or the smartbugs-curated repository) via the right-hand panel > Input > '+ Add Input'. "
        "If the file has another name, set CONFIG['DATA_CSV'] to its full path (copy it from the Input panel)."
        % sorted(NEEDED))
if resolved != C["DATA_CSV"]:
    print("DATA_CSV not found at the configured path; using", resolved)
C["DATA_CSV"] = resolved
df = pd.read_csv(resolved)
print("Loaded", resolved, "with columns", list(df.columns))
if C["COL_FUNC_LABEL"] in df.columns and df[C["COL_FUNC_LABEL"]].notna().any():
    df["y_name"] = df[C["COL_FUNC_LABEL"]].fillna(C["BENIGN"])
else:
    raise ValueError("functions.csv has no function-level label column; the report's protocol needs one.")
df = df.rename(columns={C["COL_SOURCE"]: "source", C["COL_GROUP"]: "contract_id"})
df = df[["source", "y_name", "contract_id"] + [c for c in ("func_name", "start_line", "end_line") if c in df.columns]]
df = df.dropna(subset=["source", "y_name", "contract_id"]).reset_index(drop=True)
df["contract_id"] = df["contract_id"].astype(str)
print(len(df), "functions from", df.contract_id.nunique(), "contracts")

def normalize_code(src):
    src = re.sub(r"//.*?$|/\*.*?\*/", " ", str(src), flags=re.S | re.M)
    return re.sub(r"\s+", " ", src).strip()
df["norm"] = df.source.map(normalize_code)
df["h"] = df.norm.map(lambda s: hashlib.sha256(s.encode()).hexdigest())

# --- PoC contracts: removed from the cross-validation corpus and kept aside for Step 9 ---
poc_ids = set(C["POC_CONTRACT_IDS"])
missing = [p for p in poc_ids if p not in set(df.contract_id)]
if missing: print("WARNING: PoC contract ids not found in functions.csv:", missing)
if not poc_ids: print("No PoC contracts listed in POC_CONTRACT_IDS: nothing is excluded from training.")
df_poc = df[df.contract_id.isin(poc_ids)].copy()
print("PoC functions set aside:", len(df_poc))

In [ ]:
# ===================== duplicate screening =====================
# exact duplicates of non-trivial normalised functions appearing in more than one contract
nontriv = df[df.norm.str.len() >= C["DEDUP_MIN_CHARS"]]
dup = nontriv.groupby("h").contract_id.nunique()
dup_hashes = dup[dup > 1].index
dup_rows = nontriv[nontriv.h.isin(dup_hashes)]
print(f"{len(dup_hashes)} non-trivial function bodies occur in more than one contract "
      f"({len(dup_rows)} functions, {dup_rows.contract_id.nunique()} contracts).")
label_conflicts = dup_rows.groupby("h").y_name.nunique()
print("duplicate bodies with conflicting labels:", int((label_conflicts > 1).sum()))
dup_rows.sort_values("h")[["h", "contract_id", "y_name"]].to_csv(f"{OUT}/duplicate_functions.csv", index=False)

# union-find: contracts sharing a duplicate body go into one CV group
parent = {c: c for c in df.contract_id.unique()}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
if C["DEDUP_GROUPS"]:
    for h, g in dup_rows.groupby("h"):
        cs = g.contract_id.unique()
        for c2 in cs[1:]:
            a, b = find(cs[0]), find(c2)
            if a != b: parent[b] = a
df["group"] = df.contract_id.map(find)
print("CV groups:", df.group.nunique(), "(contracts:", df.contract_id.nunique(), ")")
json.dump(dict(dup_bodies=int(len(dup_hashes)), dup_functions=int(len(dup_rows)),
               dup_contracts=int(dup_rows.contract_id.nunique()),
               conflicting=int((label_conflicts > 1).sum()), groups=int(df.group.nunique())),
          open(f"{OUT}/duplicate_screening.json", "w"), indent=2)

In [ ]:
# ===================== rare-class policy + class map =====================
cv = df[~df.contract_id.isin(poc_ids)].reset_index(drop=True)
counts = cv.y_name.value_counts()
rare = [c for c, n in counts.items() if n < C["MIN_SUPPORT"] and c != C["BENIGN"]]
print("rare classes (<%d):" % C["MIN_SUPPORT"], {c: int(counts[c]) for c in rare})
if C["RARE_POLICY"] == "merge":
    cv.loc[cv.y_name.isin(rare), "y_name"] = "other"
elif C["RARE_POLICY"] == "exclude":
    cv = cv[~cv.y_name.isin(rare)].reset_index(drop=True)
classes = sorted(cv.y_name.unique().tolist()); classes.remove(C["BENIGN"]); classes = [C["BENIGN"]] + classes
label2id = {c: i for i, c in enumerate(classes)}; id2label = {i: c for c, i in label2id.items()}
cv["y"] = cv.y_name.map(label2id); K = len(classes); BEN = label2id[C["BENIGN"]]
support = cv.y.value_counts().reindex(range(K)).fillna(0).astype(int).values
HEAD = [k for k in range(K) if support[k] >= C["MIN_SUPPORT"]]   # classes in the headline macro average
print(pd.DataFrame({"class": classes, "n": support}))
print("headline classes:", [id2label[k] for k in HEAD])

## 2. Model, loss, training with an inner validation split

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef, cohen_kappa_score,
                             precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix)

tokenizer = AutoTokenizer.from_pretrained(C["MODEL_NAME"], revision=C["MODEL_REVISION"])

class FunctionDataset(Dataset):
    def __init__(self, sources, labels):
        self.s = list(sources); self.y = list(labels)
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        enc = tokenizer(normalize_code(self.s[i]), truncation=True, max_length=C["MAX_LEN"],
                        padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"][0], "attention_mask": enc["attention_mask"][0],
                "labels": torch.tensor(int(self.y[i]))}

class CodeBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(C["MODEL_NAME"], revision=C["MODEL_REVISION"])
        self.dropout = nn.Dropout(C["DROPOUT"]); self.classifier = nn.Linear(self.backbone.config.hidden_size, K)
    def forward(self, ids, am):
        cls = self.backbone(input_ids=ids, attention_mask=am).last_hidden_state[:, 0]
        if self.training and C["MSD_K"] > 1:
            return torch.stack([self.classifier(self.dropout(cls)) for _ in range(C["MSD_K"])]).mean(0)
        return self.classifier(self.dropout(cls))

class Loss(nn.Module):
    def __init__(self, alpha):
        super().__init__(); self.alpha = alpha
    def forward(self, logits, t):
        logp = F.log_softmax(logits, 1); logpt = logp.gather(1, t.view(-1, 1)).squeeze(1)
        loss = -logpt
        if C["LOSS"] == "focal":
            loss = (1 - logpt.exp()).pow(C["FOCAL_GAMMA"]) * loss
        if self.alpha is not None:
            loss = self.alpha.to(logits.device)[t] * loss
        return loss.mean()

def make_alpha(y):
    if not C["CLASS_WEIGHTS"]: return None
    n = np.bincount(y, minlength=K).astype(float); n[n == 0] = 1
    return torch.tensor(n[BEN] / n, dtype=torch.float)          # alpha_c = N_benign / N_c

def probs_of(model, frame):
    dl = DataLoader(FunctionDataset(frame.source, frame.y), batch_size=32)
    model.eval(); out = []
    with torch.no_grad():
        for b in dl:
            out.append(torch.softmax(model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)).float(), 1).cpu().numpy())
    return np.concatenate(out)

def inner_split(tr, seed):
    # contract-grouped inner split of the training folds, stratified as far as grouping allows
    n_inner = max(2, int(round(1 / C["INNER_VAL_FRAC"])))
    sg = StratifiedGroupKFold(n_splits=n_inner, shuffle=True, random_state=seed)
    a, b = next(sg.split(tr, tr.y, groups=tr.group))
    return tr.iloc[a], tr.iloc[b]

def train_model(tr, va, seed, epochs=None):
    # va is used ONLY for early stopping; it never overlaps the outer test fold
    seed_everything(seed)
    model = CodeBERTClassifier().to(DEVICE)
    dl = DataLoader(FunctionDataset(tr.source, tr.y), batch_size=C["BATCH_SIZE"], shuffle=True)
    crit = Loss(make_alpha(tr.y.values))
    opt = torch.optim.AdamW(model.parameters(), lr=C["LR"], weight_decay=C["WEIGHT_DECAY"])
    E = epochs or C["EPOCHS"]; steps = (len(dl) // C["GRAD_ACCUM"] + 1) * E
    sch = get_cosine_schedule_with_warmup(opt, int(C["WARMUP_RATIO"] * steps), steps)
    amp = C["FP16"] and DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best, best_state, best_ep, wait, hist = -1, None, E - 1, 0, []
    for ep in range(E):
        model.train(); opt.zero_grad(); tot = 0
        for i, b in enumerate(dl):
            with torch.autocast(device_type="cuda", enabled=amp):
                loss = crit(model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)), b["labels"].to(DEVICE)) / C["GRAD_ACCUM"]
            scaler.scale(loss).backward(); tot += loss.item() * C["GRAD_ACCUM"]
            if (i + 1) % C["GRAD_ACCUM"] == 0 or i + 1 == len(dl):
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        rec = {"epoch": ep, "train_loss": tot / len(dl)}
        if va is not None:
            p = probs_of(model, va); f = f1_score(va.y, p.argmax(1), average="macro", zero_division=0)
            rec["inner_val_macro_f1"] = f
            if f > best:
                best, best_ep, wait = f, ep, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= C["PATIENCE"]: hist.append(rec); break
        hist.append(rec)
    if best_state is not None: model.load_state_dict(best_state)
    return model, hist, best_ep

## 3. Outer cross-validation (per seed)

For every outer fold the model is trained on the other folds minus an inner validation split, early
stopping and the nested thresholds use only that inner split, and the outer fold is scored once.

In [ ]:
def best_threshold(p, y, grid=np.round(np.arange(0.01, 1.00, 0.01), 2)):
    if y.sum() < 2: return 0.5          # too few positives to tune a threshold: fall back to 0.5
    f = [f1_score(y, (p >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f))])

def decide(P, thr):
    # Eq. 3.24: among vulnerability classes reaching their threshold, pick max p/t; otherwise benign
    yhat = np.full(len(P), BEN)
    for i in range(len(P)):
        best, arg = -1, None
        for c in range(K):
            if c == BEN or thr.get(c) is None: continue
            if P[i, c] >= thr[c] and P[i, c] / thr[c] > best:
                best, arg = P[i, c] / thr[c], c
        if arg is not None: yhat[i] = arg
    return yhat

RUNS = {}
for seed in C["SEEDS"]:
    sg = StratifiedGroupKFold(n_splits=C["N_SPLITS"], shuffle=True, random_state=seed)
    oof = np.zeros((len(cv), K)); fold_of = np.full(len(cv), -1); thr_nested = {}; hist = {}; best_eps = []
    for f, (tri, tei) in enumerate(sg.split(cv, cv.y, groups=cv.group)):
        tr_all, te = cv.iloc[tri], cv.iloc[tei]
        tr, va = inner_split(tr_all, seed + f)
        assert not set(va.group) & set(te.group) and not set(tr.group) & set(te.group)
        model, h, be = train_model(tr, va, seed + f)
        hist[f] = h; best_eps.append(be)
        pva = probs_of(model, va)
        thr_nested[f] = {c: best_threshold(pva[:, c], (va.y.values == c).astype(int)) for c in range(K)}
        oof[tei] = probs_of(model, te); fold_of[tei] = f
        print(f"seed {seed} fold {f}: train {len(tr)}  inner-val {len(va)}  test {len(te)}  best epoch {be}")
        del model; torch.cuda.empty_cache()
    np.save(f"{OUT}/oof_probs_{TAG}_seed{seed}.npy", oof)
    pd.DataFrame(oof, columns=classes).assign(y=cv.y_name.values, fold=fold_of, contract_id=cv.contract_id.values
        ).to_csv(f"{OUT}/oof_probs_{TAG}_seed{seed}.csv", index=False)
    RUNS[seed] = dict(oof=oof, fold=fold_of, thr_nested=thr_nested, hist=hist, best_eps=best_eps)
json.dump({s: {"best_epochs": r["best_eps"]} for s, r in RUNS.items()}, open(f"{OUT}/best_epochs_{TAG}.json", "w"))

## 4. Metrics: majority baseline first, MCC and macro-F1 as headline

In [ ]:
def metrics(y, yhat):
    m = dict(MCC=matthews_corrcoef(y, yhat),
             macro_F1_all=f1_score(y, yhat, labels=list(range(K)), average="macro", zero_division=0),
             macro_F1_head=f1_score(y, yhat, labels=HEAD, average="macro", zero_division=0),
             macro_P=precision_score(y, yhat, labels=list(range(K)), average="macro", zero_division=0),
             macro_R=recall_score(y, yhat, labels=list(range(K)), average="macro", zero_division=0),
             kappa=cohen_kappa_score(y, yhat),
             weighted_F1=f1_score(y, yhat, average="weighted", zero_division=0),
             accuracy=accuracy_score(y, yhat))
    return {k: float(v) for k, v in m.items()}

y = cv.y.values
rows = []
for seed, R in RUNS.items():
    P, fo = R["oof"], R["fold"]
    # cross-fitted thresholds (earlier method, for comparison)
    thr_cf = {f: {c: best_threshold(P[fo != f, c], (y[fo != f] == c).astype(int)) for c in range(K)} for f in range(C["N_SPLITS"])}
    pred = {"argmax": P.argmax(1), "nested_thr": np.zeros(len(y), int), "crossfit_thr": np.zeros(len(y), int)}
    for f in range(C["N_SPLITS"]):
        s = fo == f
        pred["nested_thr"][s] = decide(P[s], R["thr_nested"][f])
        pred["crossfit_thr"][s] = decide(P[s], thr_cf[f])
    R["pred"] = pred; R["thr_cf"] = thr_cf
    rows.append(dict(seed=seed, rule="majority_baseline", scope="pooled", **metrics(y, np.full(len(y), BEN))))
    for rule, yh in pred.items():
        rows.append(dict(seed=seed, rule=rule, scope="pooled", **metrics(y, yh)))
        for f in range(C["N_SPLITS"]):
            s = fo == f
            rows.append(dict(seed=seed, rule=rule, scope=f"fold{f}", **metrics(y[s], yh[s])))
M = pd.DataFrame(rows); M.to_csv(f"{OUT}/metrics_{TAG}.csv", index=False)
pooled = M[M.scope == "pooled"].drop(columns=["seed", "scope"]).groupby("rule").agg(["mean", "std"])
folds = M[M.scope != "pooled"].drop(columns=["seed", "scope"]).groupby("rule").agg(["mean", "std"])
pooled.to_csv(f"{OUT}/table_pooled_{TAG}.csv"); folds.to_csv(f"{OUT}/table_folds_{TAG}.csv")
cols = ["MCC", "macro_F1_all", "macro_F1_head", "macro_P", "macro_R", "kappa", "weighted_F1", "accuracy"]
print("Pooled OOF (mean over seeds):"); print(pooled.xs("mean", axis=1, level=1)[cols].round(3))
print("\nPer-fold (mean and std over folds x seeds):"); print(folds.xs("mean", axis=1, level=1)[cols].round(3))

In [ ]:
# ===================== per-class: support, P/R/F1, ROC-AUC and AP; one-vs-rest threshold F1 =====================
per = []
for seed, R in RUNS.items():
    P, fo = R["oof"], R["fold"]; yh = P.argmax(1)
    for c in range(K):
        yc = (y == c).astype(int)
        nested = np.zeros(len(y), int); cf = np.zeros(len(y), int)
        for f in range(C["N_SPLITS"]):
            s = fo == f
            tn = R["thr_nested"][f][c]; tc = R["thr_cf"][f][c]
            nested[s] = (P[s, c] >= (tn if tn is not None else 2)).astype(int)
            cf[s] = (P[s, c] >= (tc if tc is not None else 2)).astype(int)
        per.append(dict(seed=seed, cls=id2label[c], support=int(yc.sum()), headline=c in HEAD,
            precision=precision_score(yc, (yh == c).astype(int), zero_division=0),
            recall=recall_score(yc, (yh == c).astype(int), zero_division=0),
            f1_default=f1_score(yc, (yh == c).astype(int), zero_division=0),
            f1_nested_ovr=f1_score(yc, nested, zero_division=0),
            f1_crossfit_ovr=f1_score(yc, cf, zero_division=0),
            roc_auc=roc_auc_score(yc, P[:, c]) if 0 < yc.sum() < len(yc) else np.nan,
            ap=average_precision_score(yc, P[:, c]) if yc.sum() > 0 else np.nan,
            ap_chance=yc.mean()))
PC = pd.DataFrame(per); PC.to_csv(f"{OUT}/per_class_{TAG}.csv", index=False)
print(PC.groupby("cls").mean(numeric_only=True).round(3))

In [ ]:
# ===================== confusion matrix, calibration, figures =====================
import matplotlib.pyplot as plt
s0 = C["SEEDS"][0]; R = RUNS[s0]
for rule in ("argmax", "nested_thr"):
    cm = confusion_matrix(y, R["pred"][rule], labels=list(range(K)))
    pd.DataFrame(cm, index=classes, columns=classes).to_csv(f"{OUT}/confusion_{rule}_{TAG}_seed{s0}.csv")
    fig, ax = plt.subplots(figsize=(6.5, 5.5)); ax.imshow(cm, cmap="Blues")
    for i in range(K):
        for j in range(K): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(K)); ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(K)); ax.set_yticklabels(classes, fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Confusion matrix ({rule}, OOF, seed {s0})")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_confusion_{rule}_{TAG}.png", dpi=200); plt.close()
ben = cm_arg = confusion_matrix(y, R["pred"]["argmax"], labels=list(range(K)))
print(f"benign predicted as vulnerable: {cm_arg[BEN].sum() - cm_arg[BEN, BEN]} of {cm_arg[BEN].sum()}")

def ece(P, y, bins=10):
    conf, pred = P.max(1), P.argmax(1); e = 0
    for b in range(bins):
        m = (conf > b / bins) & (conf <= (b + 1) / bins)
        if m.any(): e += abs((pred[m] == y[m]).mean() - conf[m].mean()) * m.mean()
    return e
eces = {s: float(ece(RUNS[s]["oof"], y)) for s in RUNS}; print("ECE per seed:", eces)
json.dump(eces, open(f"{OUT}/ece_{TAG}.json", "w"))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for f, h in R["hist"].items():
    hh = pd.DataFrame(h); ax[0].plot(hh.epoch, hh.train_loss, label=f"fold {f+1}")
    ax[1].plot(hh.epoch, hh.inner_val_macro_f1, label=f"fold {f+1}")
ax[0].set_title("Training loss"); ax[1].set_title("Inner-validation macro-F1"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_training_curves_{TAG}.png", dpi=200); plt.close()

## 5. Ablation

Run the notebook a second time with `RUN_TAG="ce_unweighted"`, `LOSS="ce"`, `CLASS_WEIGHTS=False`
(same seeds). Then run the cell below to compare the two runs on identical splits.

In [ ]:
runs = sorted(glob.glob(f"{OUT}/metrics_*.csv"))
if len(runs) > 1:
    comp = []
    for r in runs:
        m = pd.read_csv(r); m = m[(m.scope == "pooled") & (m.rule == "argmax")]
        comp.append(m.drop(columns=["seed", "rule", "scope"]).mean().rename(Path(r).stem.replace("metrics_", "")))
    print(pd.DataFrame(comp).round(3))
else:
    print("Only one run found; run the ablation configuration to compare.")

## 6. Model for the three-engine run (PoC contracts excluded) and CodeBERT findings

The model is trained on every contract except the three PoC contracts, for a fixed number of epochs
(the median best epoch from Step 3), so no PoC function influences training or model selection.
Thresholds are the median of the nested per-fold thresholds. Findings are written in the common
schema so the aggregator can fuse them with the Slither and SmartCheck findings.

In [ ]:
SEV = {"reentrancy": "critical", "access_control": "critical", "arithmetic": "high", "bad_randomness": "high",
       "unchecked_low_level_calls": "medium", "denial_of_service": "medium", "front_running": "medium",
       "time_manipulation": "medium"}
CANON = {"unchecked_low_level_calls": "unchecked_calls"}   # match the static normaliser's vocabulary

all_eps = [e for r in RUNS.values() for e in r["best_eps"]]
E_FINAL = int(np.median(all_eps)) + 1
thr_final = {}
for c in range(K):
    v = [r["thr_nested"][f][c] for r in RUNS.values() for f in r["thr_nested"] if r["thr_nested"][f][c] is not None]
    thr_final[c] = float(np.median(v)) if v else None
print("final epochs:", E_FINAL, " thresholds:", {id2label[c]: t for c, t in thr_final.items()})

final, _, _ = train_model(cv, None, C["SEEDS"][0], epochs=E_FINAL)    # cv already excludes the PoC contracts
ED = C["EXPORT_DIR"]
final.backbone.save_pretrained(ED); tokenizer.save_pretrained(ED)
torch.save(final.classifier.state_dict(), f"{ED}/classifier_head.pt"); torch.save(final.state_dict(), f"{ED}/full_model.pt")
json.dump(dict(architecture="CodeBERTClassifier", pooling="cls", num_labels=K, label2id=label2id,
               id2label={int(k): v for k, v in id2label.items()}, max_len=C["MAX_LEN"], loss=C["LOSS"],
               class_weights=C["CLASS_WEIGHTS"], epochs=E_FINAL, excluded_contracts=sorted(poc_ids),
               thresholds={id2label[c]: t for c, t in thr_final.items()},   # real thresholds, not F1 values
               decision_rule="Eq. 3.24: argmax p/t over classes with p >= t, else benign",
               versions=VERSIONS), open(f"{ED}/model_metadata.json", "w"), indent=2)
print("exported:", os.listdir(ED))

In [ ]:
# ---- CodeBERT findings on the PoC contracts (common schema) ----
def extract_functions(sol):
    # simple brace-matching extractor: function name, start/end line, source
    out = []
    for m in re.finditer(r"\b(function\s+(\w+)|function\s*\(|constructor\s*\(|modifier\s+(\w+))", sol):
        i = sol.find("{", m.end()); semi = sol.find(";", m.end())
        if i == -1 or (semi != -1 and semi < i): continue      # declaration without a body
        depth, j = 0, i
        while j < len(sol):
            depth += sol[j] == "{"; depth -= sol[j] == "}"
            if depth == 0: break
            j += 1
        name = m.group(2) or m.group(3) or ("constructor" if "constructor" in m.group(1) else "fallback")
        out.append(dict(func_name=name, start_line=sol.count("\n", 0, m.start()) + 1,
                        end_line=sol.count("\n", 0, j) + 1, source=sol[m.start(): j + 1]))
    return out

findings = []
sol_files = sorted(glob.glob(os.path.join(C["POC_DIR"], "**", "*.sol"), recursive=True))
if not sol_files:
    sol_files = sorted(glob.glob("/kaggle/input/**/*.sol", recursive=True))
    if sol_files: print("POC_DIR not found; using .sol files found under /kaggle/input:", sol_files)
if not sol_files:
    print("No PoC .sol files attached: skipping CodeBERT findings. Attach a dataset with the three PoC "
          "contracts (or set CONFIG['POC_DIR']) and re-run this cell only.")
for path in sol_files:
    sol = open(path, encoding="utf-8", errors="ignore").read()
    cid = hashlib.sha256(normalize_code(sol).encode()).hexdigest()
    funcs = extract_functions(sol)
    if not funcs: continue
    fr = pd.DataFrame(funcs).assign(y=0)
    P = probs_of(final, fr); yhat = decide(P, thr_final)
    for k, row in fr.iterrows():
        c = int(yhat[k])
        if c == BEN: continue
        name = id2label[c]
        findings.append(dict(contract_id=cid, contract_file=os.path.basename(path), vuln_type=CANON.get(name, name),
            severity=SEV.get(name, "low"), location=dict(function=row.func_name, start_line=int(row.start_line), end_line=int(row.end_line)),
            confidence=round(float(P[k, c]), 4), source="codebert", evidence_ref=f"codebert:{name}:p={P[k, c]:.3f}:t={thr_final[c]}"))
json.dump(findings, open(f"{OUT}/codebert_findings_poc.json", "w"), indent=2)
print(len(findings), "CodeBERT findings written to codebert_findings_poc.json")
pd.DataFrame(findings)

## 7. Three-engine run (outside this notebook)

1. Copy `codebert_findings_poc.json` and the `codebert-smartbugs-v3/` folder into the framework
   repository (the `nlp_service` loads `full_model.pt` and `model_metadata.json`).
2. Run Slither and SmartCheck on the same three `.sol` files through the static-analysis service
   (same compiler selection as Section 3.6.1) and normalise their findings.
3. Feed all three engines' findings to the aggregator; export the clusters (type, function, engines,
   consensus status, risk score with its inputs s, c_max, m).
4. Score each engine and the aggregated output against the five SmartBugs labels with the versioned
   matching script (strict and lenient).
5. Submit every cluster with risk at or above the review threshold (or at least the top cluster per
   contract) to the testnet validator; save each outcome and trace.
6. Optionally repeat steps 2 to 4 for all 143 SmartBugs Curated contracts with the static analyzers
   (CodeBERT findings on those contracts must come from the out-of-fold predictions saved above, never
   from a model trained on them).

## 8. Where the outputs go in the report

| Output | Report location |
|---|---|
| `table_pooled_*.csv`, `table_folds_*.csv` | Tables 4.1, 4.2, C.1, C.2 (MCC and macro-F1 first; baseline column) |
| `per_class_*.csv` | Tables 4.3, 4.4, D.1 (use `f1_nested_ovr`; overall metrics after the decision rule come from `nested_thr` rows) |
| `confusion_*`, `fig_*` | Figures 4.1, 4.3, 4.6 |
| `duplicate_screening.json` | Section 3.4 (replace "no duplicate screening was performed") |
| `ece_*.json` | Section 4.7 |
| metrics of the `ce_unweighted` run | new ablation paragraph in Section 4.2 |
| `codebert_findings_poc.json` + aggregator/testnet outputs | Sections 4.10 to 4.14 (three-engine results) |
| `environment.json`, `model_metadata.json` | Appendix E |